⏱️ **Time required:** ~3 minutes | **Type:** Domain pipeline (run all cells)

# 🚗 RideFlow Marketplace — Domain Pipeline

**Domain Owner:** Marketplace Engineering  
**System:** rideflow  

This notebook represents the **Marketplace domain team's** autonomous pipeline. It generates synthetic ride-sharing data, processes it through the full Medallion architecture (Bronze → Silver → Gold), and publishes **Data Products** that other domains can consume.

| Published Data Product | Layer | Description |
| :-- | :-- | :-- |
| `silver_rideflow_trips` | Silver | Deduplicated, typed trip fact table |
| `silver_rideflow_driver_profiles` | Silver | Cleaned driver dimension |
| `silver_rideflow_rider_profiles` | Silver | Cleaned rider dimension |
| `gold_rideflow_trip_kpis` | Gold | Revenue, volume, distance KPIs |
| `gold_rideflow_rider_lifetime_value` | Gold | Rider LTV analysis |
| `gold_rideflow_surge_pricing_model` | Gold | Dynamic surge multipliers |

---
## Step 1 · Environment Setup

Install dependencies, detect runtime (Colab vs local), and configure paths.

In [1]:
import os
import subprocess
import sys

# ── Install lakelogic ─────────────────────────────────────────────────
# Update the path below to match your local lakelogic checkout.
# On Colab (or if the path doesn't exist), falls back to PyPI.
_LAKELOGIC_LOCAL = r"C:\_Personal\_SaaS\lakelogic"

if os.path.isdir(_LAKELOGIC_LOCAL):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", _LAKELOGIC_LOCAL, "-q"])
    print(f"\u2705 Installed lakelogic (editable) from {_LAKELOGIC_LOCAL}")
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lakelogic", "-q"])
    print("\u2705 Installed lakelogic from PyPI")

✅ Installed lakelogic (editable) from C:\_Personal\_SaaS\lakelogic


In [2]:
import subprocess
import sys
import importlib
import os

# Install LakeLogic + dashboard dependencies if missing
for pkg in ["lakelogic", "panel", "hvplot", "pyparsing"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", f"{pkg}[polars]" if pkg == "lakelogic" else pkg]
        )

# Detect environment
IN_COLAB = "google.colab" in sys.modules
print(f"Running in: {'Google Colab' if IN_COLAB else 'Local'}")


import _setup as s

Running in: Local
lakelogic v1.21.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


In [3]:
import panel as pn
import polars as pl
import lakelogic as ll
from pathlib import Path
from datetime import datetime, timezone

# Panel init — use colab comms if in Colab
if IN_COLAB:
    pn.extension("tabulator", comms="colab")
else:
    pn.extension("tabulator", design="bootstrap")

# Paths
DOMAIN_ROOT = Path("assets/domains_rideflow/marketplace/rideflow")
SYSTEM_YAML = str(DOMAIN_ROOT / "_system.yaml")
LAKEHOUSE = Path("./lakehouse")
LANDING_ROOT = LAKEHOUSE / "_data" / "landing_marketplace" / "rideflow"
os.environ["LAKELOGIC_OBSERVATORY_ENDPOINT"] = "http://127.0.0.1:3007/api/v1/operations/run-logs/ingest"
os.environ["LAKELOGIC_API_KEY"] = "llc_sk_9bdfa73e785acdaeb19ac23ee4f257b0"

# os.env['  endpoint: "${LAKELOGIC_OBSERVATORY_ENDPOINT}"  # e.g., "${LAKELOGIC_OBSERVATORY_ENDPOINT}"
#   api_key: "${LAKELOGIC_API_KEY}"   # e.g., "${LAKELOGIC_API_KEY}" for prod']

print(f"lakelogic v{ll.__version__}")
print(f"Lakehouse: {LAKEHOUSE.resolve()}")

lakelogic v1.21.0
Lakehouse: C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse


---
## Step 2 · Load the Data Contract Registry

The `_system.yaml` registry is the single source of truth for this entire platform.
It declares every entity, its schema contract, lineage rules, materialization strategy, and cost model.

> Changing the `environment` parameter is all it takes to switch between local prototyping and Azure production.

In [4]:
from lakelogic.core.registry import DomainRegistry

ENV = "colab" if IN_COLAB else "local"
registry = DomainRegistry.from_yaml(SYSTEM_YAML, environment=ENV, storage_mode="direct")

print(f"Domain:  {registry.domain}")
print(f"System:  {registry.system}")
print(f"Env:     {ENV}")
print(f"\nContracts loaded: {len(registry.get_active_contracts())}")
for c in registry.get_active_contracts():
    print(f"  [{c.layer:6s}] {c.entity}")

2026-04-30 20:23:25.565 | INFO     | lakelogic.core.registry:from_yaml:404 - Domain config inherited from assets\domains_rideflow\marketplace\_domain.yaml
2026-04-30 20:23:25.566 | INFO     | lakelogic.core.registry:_validate_observatory_config:289 - ✅ Observatory (_system.yaml): config validated — endpoint=http://127.0.0.1:3007/api/v1/operations/run-logs/ingest, emit_on=['success', 'partial', 'failed'], environments=['dev', 'prod', 'staging', 'local']
2026-04-30 20:23:25.571 | INFO     | lakelogic.core.registry:_validate_observatory_config:289 - ✅ Observatory (Contract: rider_profiles): config validated — endpoint=http://127.0.0.1:3007/api/v1/operations/run-logs/ingest, emit_on=['success', 'partial', 'failed'], environments=['dev', 'prod', 'staging', 'local']
2026-04-30 20:23:25.576 | INFO     | lakelogic.core.registry:_validate_observatory_config:289 - ✅ Observatory (Contract: rider_app_events): config validated — endpoint=http://127.0.0.1:3007/api/v1/operations/run-logs/ingest, emit

Domain:  marketplace
System:  rideflow
Env:     local

Contracts loaded: 15
  [bronze] rider_profiles
  [bronze] rider_app_events
  [bronze] driver_profiles
  [bronze] driver_telemetry
  [bronze] trip_requests
  [bronze] trip_completed
  [bronze] trip_cancellations
  [silver] silver_rideflow_driver_profiles
  [silver] silver_rideflow_driver_telemetry
  [silver] silver_rideflow_rider_profiles
  [silver] silver_rideflow_trips
  [gold  ] gold_rideflow_trip_kpis
  [gold  ] gold_rideflow_driver_scorecard
  [gold  ] gold_rideflow_rider_daily_metrics
  [gold  ] gold_rideflow_surge_pricing_model


---
## Step 3 · Pipeline Lineage DAG

An interactive topological map of the RideFlow Data Mesh, automatically generated from the declarative `depends_on` rules in the Data Contracts.

In [5]:
from IPython.display import HTML
from lakelogic.pipeline.runner import LakehousePipeline

pipeline = LakehousePipeline(registry)
display(HTML(pipeline.visualize_dag(title="")))

2026-04-30 20:23:30.952 | DEBUG    | lakelogic.pipeline.runner:__init__:177 - Disabled DeletionVectors by default in Spark session for OSS compatibility.


---
## Step 4 · Generate Streaming Data

The `StreamingSimulator` generates **24 hours** of realistic ride-sharing data:
- **Dimension entities** (riders, drivers) with stable identity pools
- **Fact entities** (trip requests → completions → cancellations) following hour-of-day demand curves
- **Event entities** (telemetry, app events) proportional to active trips

All entities maintain **referential integrity** — every `rider_id` in a trip exists in the rider pool.

In [6]:
from lakelogic.core.streaming import StreamingSimulator
from datetime import timedelta

sim = StreamingSimulator.rideflow_marketplace(
    landing_root=str(LANDING_ROOT),
    window_minutes=60,
    start_time=datetime.now(timezone.utc) - timedelta(days=3),
    seed=42,
    initial_riders=200,
    initial_drivers=100,
)

print("🚀 Streaming simulator configured")
print(f"   Landing root: {LANDING_ROOT}")
print("   Window size:  60 minutes")
print("   Entities:     7 (2 dimensions, 3 facts, 2 events)")

🚀 Streaming simulator configured
   Landing root: lakehouse\_data\landing_marketplace\rideflow
   Window size:  60 minutes
   Entities:     7 (2 dimensions, 3 facts, 2 events)


In [7]:
# Generate 24 hours of simulated streaming data
NUM_WINDOWS = 24

total_rows = 0
windows = []
for window in sim.run(num_windows=NUM_WINDOWS, micro_batches=10, up_to=datetime.now(timezone.utc), resume=True):
    windows.append(window)
    total_rows += window.total_rows

print(f"\n{'=' * 60}")
if not windows:
    print("✅ No new windows generated. The simulator has caught up to the current time (up_to constraint reached).")
else:
    print(f"✅ Generated {total_rows:,} total rows across {len(windows)} windows")
    print(f"   Seed window:    {windows[0].total_rows:,} rows (initial pool)")
    print(f"   Avg per window: {total_rows // len(windows):,} rows")

# FK consistency check
fk = sim.validate_fk_consistency()
print("\n🔗 FK Consistency:")
print(f"   Riders:    {fk['total_riders']}")
print(f"   Drivers:   {fk['total_drivers']}")
print(f"   Healthy:   {'✅' if fk['fk_pools_healthy'] else '❌'}")

2026-04-30 20:23:30.988 | INFO     | lakelogic.core.streaming:_rebuild_state_from_landing:805 - No existing landing zone found -- starting fresh
2026-04-30 20:23:31.001 | INFO     | lakelogic.core.streaming:_seed_initial_pools:674 - 🌱 Seeded initial pools: 200 riders, 100 drivers
2026-04-30 20:23:31.050 | INFO     | lakelogic.core.streaming:run:771 - ⏱ Window   0 | 2026-04-27 19:23 |   727 rows | rider_profiles=6 driver_profiles=2 trip_requests=92 trip_completed=55 trip_cancellations=9 driver_telemetry=412 rider_app_events=151 | 10 micro-batches/entity
2026-04-30 20:23:31.090 | INFO     | lakelogic.core.streaming:run:771 - ⏱ Window   1 | 2026-04-27 20:23 |   639 rows | rider_profiles=6 driver_profiles=2 trip_requests=67 trip_completed=51 trip_cancellations=7 driver_telemetry=387 rider_app_events=119 | 10 micro-batches/entity
2026-04-30 20:23:31.129 | INFO     | lakelogic.core.streaming:run:771 - ⏱ Window   2 | 2026-04-27 21:23 |   609 rows | rider_profiles=6 driver_profiles=2 trip_requ


✅ Generated 13,994 total rows across 25 windows
   Seed window:    300 rows (initial pool)
   Avg per window: 559 rows

🔗 FK Consistency:
   Riders:    391
   Drivers:   148
   Healthy:   ✅


In [8]:
# Verify partition structure
import glob

for entity in ["trip_requests", "trip_completed", "rider_profiles"]:
    files = glob.glob(str(LANDING_ROOT / entity / "**/*.csv"), recursive=True)
    print(f"  {entity}: {len(files)} partition files")
    if files:
        # Show first and last partition path
        files.sort()
        print(f"    First: {files[0].split(entity)[-1]}")
        print(f"    Last:  {files[-1].split(entity)[-1]}")

  trip_requests: 235 partition files
    First: \y_2026\m_04\d_27\h_19\batch_00_0d39d6.csv
    Last:  \y_2026\m_04\d_28\h_18\batch_09_daffdc.csv
  trip_completed: 219 partition files
    First: \y_2026\m_04\d_27\h_19\batch_00_9da49f.csv
    Last:  \y_2026\m_04\d_28\h_18\batch_09_664160.csv
  rider_profiles: 182 partition files
    First: \y_2026\m_04\d_27\h_19\batch_00_1ef399.csv
    Last:  \y_2026\m_04\d_28\h_18\batch_05_227693.csv


---
## Step 4.5 · Inject Deterministic Edge Cases (TC-001 to TC-008)

The StreamingSimulator above generated realistic, time-aware trip data with organic variance.
Now we surgically inject **exactly 800 edge-case rows** into the `trip_completed` landing directory.
These represent 8 specific failure modes that the pipeline's quarantine layer must catch:

| TC | Description | Rows | What the pipeline should catch |
| :-- | :-- | --: | :-- |
| TC-001 | Negative fare | 120 | Business logic: `fare_amount >= 0` |
| TC-002 | driver_id = rider_id | 80 | Self-service fraud detection |
| TC-003 | Dropoff before pickup | 150 | Temporal sanity: `dropoff_at > pickup_at` |
| TC-004 | Distance > 500km | 100 | Geospatial bounds check |
| TC-005 | SQL injection in notes | 150 | Security / input sanitization |
| TC-006 | Surge > 5.0x | 80 | Cap violation: `surge_multiplier <= 5.0` |
| TC-007 | Unknown city code | 60 | Referential integrity check |
| TC-008 | Duplicate trip_id | 60 | Idempotency / uniqueness constraint |

After the pipeline runs, we expect:
```
source_count = good_count + bad_count
"100% reconciliation. Every row accounted for."
```

In [9]:
# ── Deterministic Edge-Case Injection ────────────────────────────────────
# Injects exactly 800 mutated rows into the trip_completed landing zone
# so the pipeline quarantine layer can demonstrate 100% reconciliation.

import uuid
import random
from datetime import datetime, timedelta, timezone

random.seed(42)
NUM_EDGE_CASES = 800

CITY_PROFILES = {
    "LON": {"timezone": "Europe/London", "base_trips_per_hour": 500},
    "NYC": {"timezone": "US/Eastern", "base_trips_per_hour": 800},
    "BER": {"timezone": "Europe/Berlin", "base_trips_per_hour": 200},
    "PAR": {"timezone": "Europe/Paris", "base_trips_per_hour": 350},
    "TYO": {"timezone": "Asia/Tokyo", "base_trips_per_hour": 600},
    "SYD": {"timezone": "Australia/Sydney", "base_trips_per_hour": 150},
}

cities = list(CITY_PROFILES.keys())
base_time = datetime(2026, 4, 15, 8, 0, tzinfo=timezone.utc)

# Generate 800 base rows with valid structure
records = []
for i in range(NUM_EDGE_CASES):
    trip_id = str(uuid.uuid4())
    rider_id = f"R-{uuid.uuid4().hex[:8]}"
    driver_id = f"D-{uuid.uuid4().hex[:8]}"
    city = random.choice(cities)
    hr = random.choices(range(24), weights=[1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3, 4, 4, 3, 1, 1, 1])[0]
    req_dt = base_time.replace(hour=hr, minute=random.randint(0, 59))
    pickup_dt = req_dt + timedelta(minutes=random.randint(1, 10))
    dropoff_dt = req_dt + timedelta(minutes=random.randint(15, 60))

    records.append(
        {
            "trip_id": trip_id,
            "rider_id": rider_id,
            "driver_id": driver_id,
            "trip_type": random.choice(["ride", "eats_delivery"]),
            "pickup_lat": str(round(random.uniform(51.4, 51.6), 6)),
            "pickup_lng": str(round(random.uniform(-0.2, 0.1), 6)),
            "dropoff_lat": str(round(random.uniform(51.4, 51.6), 6)),
            "dropoff_lng": str(round(random.uniform(-0.2, 0.1), 6)),
            "city_code": city,
            "requested_at": req_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "pickup_at": pickup_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "dropoff_at": dropoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "distance_km": str(round(random.uniform(1, 25), 1)),
            "duration_minutes": str(random.randint(5, 45)),
            "fare_amount": str(round(random.uniform(5, 80), 2)),
            "surge_multiplier": str(round(random.uniform(1.0, 3.5), 2)),
            "tip_amount": str(round(random.uniform(0, 10), 2)),
            "payment_method": random.choice(["card", "apple_pay", "google_pay", "cash"]),
            "rider_rating": str(random.randint(1, 5)),
            "driver_rating": str(random.randint(1, 5)),
            "notes": "",
        }
    )

# ── Apply TC-001 to TC-008 mutations ──────────────────────────────────────
for i in range(0, 120):
    records[i]["fare_amount"] = "-15.50"  # TC-001: Negative fare
for i in range(120, 200):
    records[i]["driver_id"] = records[i]["rider_id"]  # TC-002: Self-service fraud
for i in range(200, 350):  # TC-003: Time travel
    records[i]["pickup_at"] = "2026-04-15T12:00:00Z"
    records[i]["dropoff_at"] = "2026-04-15T11:00:00Z"
for i in range(350, 450):
    records[i]["distance_km"] = "650.5"  # TC-004: Impossible trip
for i in range(450, 600):
    records[i]["notes"] = "DROP TABLE trips; SELECT * FROM users;"  # TC-005
for i in range(600, 680):
    records[i]["surge_multiplier"] = "5.8"  # TC-006: Cap violation
for i in range(680, 740):
    records[i]["city_code"] = "XYZ"  # TC-007: Unknown market
dup_id = records[0]["trip_id"]  # Reuse first trip's ID
for i in range(740, 800):
    records[i]["trip_id"] = dup_id  # TC-008: Duplicate ID

random.shuffle(records)
edge_df = pl.DataFrame(records)

# Write to the trip_completed landing directory as a separate partition file
edge_dir = LANDING_ROOT / "trip_completed" / "y_2026" / "m_04" / "d_15" / "h_99"
edge_dir.mkdir(parents=True, exist_ok=True)
edge_file = edge_dir / "edge_cases.csv"
edge_df.write_csv(str(edge_file))

print(f"\u2705 Injected {NUM_EDGE_CASES} deterministic edge cases into landing zone")
print(f"   File: {edge_file}")
print()
print("   TC-001  Negative fare:         120 rows")
print("   TC-002  driver_id=rider_id:      80 rows")
print("   TC-003  Dropoff before pickup:  150 rows")
print("   TC-004  Distance > 500km:       100 rows")
print("   TC-005  SQL injection in notes: 150 rows")
print("   TC-006  Surge > 5.0x:            80 rows")
print("   TC-007  Unknown city code:        60 rows")
print("   TC-008  Duplicate trip_id:        60 rows")
print(f"   {'=' * 45}")
print(f"   TOTAL edge cases:               {NUM_EDGE_CASES} rows")

✅ Injected 800 deterministic edge cases into landing zone
   File: lakehouse\_data\landing_marketplace\rideflow\trip_completed\y_2026\m_04\d_15\h_99\edge_cases.csv

   TC-001  Negative fare:         120 rows
   TC-002  driver_id=rider_id:      80 rows
   TC-003  Dropoff before pickup:  150 rows
   TC-004  Distance > 500km:       100 rows
   TC-005  SQL injection in notes: 150 rows
   TC-006  Surge > 5.0x:            80 rows
   TC-007  Unknown city code:        60 rows
   TC-008  Duplicate trip_id:        60 rows
   TOTAL edge cases:               800 rows


---
## Pipeline Config

- **retry_attempts**: Max retries if a dataset fails due to temporary network or storage glitches.
- **retry_base_wait_seconds**: Initial wait time before retrying. Automatically scales up (e.g., 5s, 10s, 20s) to avoid overwhelming the system.
- **parallel**: Processes independent datasets simultaneously across multiple CPU threads for much faster ingestion.
- **engine**: Defines the compute backend (`polars` for local speed, `spark` for distributed clusters).
- **reload_layers**: Forces a complete historical wipe and reload of specific layers (e.g., `"silver,gold"`).


In [10]:
retry_attempts = 1
parallel = False
retry_base_wait_seconds = 5
engine = "duckdb"
reload_layers = ""
enable_incremental_streaming = False

---
## Step 5 · Bronze Ingestion

Ingest the streaming landing data into Bronze Delta tables. Every row gets lineage columns (`_lakelogic_source`, `_lakelogic_processed_at`, `_lakelogic_run_id`) automatically injected.

In [11]:
from lakelogic.pipeline.runner import LakehousePipeline
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

print("Starting Bronze Ingestion via LakeLogic Pipeline Driver...")
runner = LakehousePipeline(registry, engine=engine)

summary = runner.run(
    target_layers="bronze",
    reload_layers=reload_layers,
    dry_run=False,
    environment=ENV,
    retry_attempts=retry_attempts,
    retry_base_wait_seconds=retry_base_wait_seconds,
    parallel=False,
)
print(summary)

2026-04-30 20:23:32.314 | INFO     | lakelogic.pipeline.runner:run:1505 - Pipeline storage mode: direct
2026-04-30 20:23:32.317 | INFO     | lakelogic.pipeline.runner:run:1606 - ── Processing Layer: BRONZE (7 contracts) ──


Starting Bronze Ingestion via LakeLogic Pipeline Driver...


2026-04-30 20:23:32.320 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2069 -   ─────────────────────────────────────────────────────────
2026-04-30 20:23:32.321 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2070 -   📄 [bronze] rider_profiles | Contract: Bronze — RideFlow Rider Profiles v1.0.0
2026-04-30 20:23:32.384 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\_data\landing_marketplace\rideflow\rider_profiles via duckdb
2026-04-30 20:23:32.745 | INFO     | lakelogic.core.processor:run_source:1441 - Initial load detected (no prior watermark) — scanning all partitions (set lookback_days to limit)
2026-04-30 20:23:33.093 | WARNING  | lakelogic.core.masking_engine:apply:377 - PII fields detected without masking strategy: [name, email, phone, date_of_birth, home_address, city_code]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for th

 PIPELINE RUN SUMMARY
  Pipeline run    : c6bcb99d-a869-41c8-a3b2-69098a1abaf0
  Environment     : local
  Dry run         : False

  Table Name                             Layer    Status     Rows     Good/Qrtn   
  ------------------------------------------------------------------------------
  bronze_rideflow_rider_profiles         bronze   success    391      391/0       
  bronze_rideflow_rider_app_events       bronze   success    2869     2869/0      
  bronze_rideflow_driver_profiles        bronze   success    148      148/0       
  bronze_rideflow_driver_telemetry       bronze   success    7916     7916/0      
  bronze_rideflow_trip_requests          bronze   success    1453     1453/0      
  bronze_rideflow_trip_completed         bronze   success    1838     1838/0      
  bronze_rideflow_trip_cancellations     bronze   success    179      179/0       


---
## Step 6 · Silver Processing

Silver layer: type casting, deduplication, and merge. Schema enforcement quarantines rows that don't match the contract — no silent data corruption.

In [12]:
print("Starting Silver Processing via LakeLogic Pipeline Driver...")

summary = runner.run(
    target_layers="silver",
    reload_layers=reload_layers,
    dry_run=False,
    environment=ENV,
    retry_attempts=retry_attempts,
    retry_base_wait_seconds=retry_base_wait_seconds,
    parallel=parallel,
)
print(summary)

2026-04-30 20:23:37.455 | INFO     | lakelogic.pipeline.runner:run:1505 - Pipeline storage mode: direct
2026-04-30 20:23:37.459 | INFO     | lakelogic.pipeline.runner:run:1606 - ── Processing Layer: SILVER (4 contracts) ──
2026-04-30 20:23:37.461 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2069 -   ─────────────────────────────────────────────────────────
2026-04-30 20:23:37.461 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2070 -   📄 [silver] silver_rideflow_driver_profiles | Contract: Silver — Rideflow Driver Profiles v1.0.0
2026-04-30 20:23:37.486 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_driver_profiles via duckdb
2026-04-30 20:23:37.554 | INFO     | lakelogic.core.processor:run_source:1815 - Incremental load: first run or target empty — loading all rows.


Starting Silver Processing via LakeLogic Pipeline Driver...


2026-04-30 20:23:37.692 | WARNING  | lakelogic.core.masking_engine:apply:377 - PII fields detected without masking strategy: [name, email, phone, date_of_birth, home_address, licence_number, licence_plate, bank_account_last_four, city_code]. Set 'masking:' (nullify|hash|redact|partial|encrypt) in your contract to enable masking for these fields.
2026-04-30 20:23:37.692 | INFO     | lakelogic.core.masking_engine:apply:384 - No PII fields with explicit masking strategy — skipping masking.
2026-04-30 20:23:37.693 | INFO     | lakelogic.core.processor:run:818 - Run complete [domain=marketplace, system=rideflow, layer=silver] | Source: 148 | Total: 148 | Good: 148 | Quarantine: 0 | Ratio: 0.00%
2026-04-30 20:23:37.721 | INFO     | lakelogic.core.materialization:materialize_dataframe:3686 - 
Casting 1 column(s) to match contract types before Delta write (new table):
Column Name                         | From Type       | To Type        
-------------------------------------------------------

 PIPELINE RUN SUMMARY
  Pipeline run    : c6bcb99d-a869-41c8-a3b2-69098a1abaf0
  Environment     : local
  Dry run         : False

  Table Name                             Layer    Status     Rows     Good/Qrtn   
  ------------------------------------------------------------------------------
  silver_rideflow_driver_profiles        silver   success    148      148/0       
  silver_rideflow_rider_profiles         silver   success    391      391/0       
  silver_rideflow_driver_telemetry       silver   success    7916     7916/0      
  silver_rideflow_trips                  silver   success    1718     1718/120    


---
## Step 6.5 · Gold Processing (Business KPIs & ML Models)

Gold layer: where the business value is unlocked! Generates Trip KPIs, Rider LTV (Lifetime Value), and the AI-driven Surge Pricing data feeds.

In [13]:
# Generate surge inference landing data for the Surge Pricing ML Model contract
import polars as pl
import random
import uuid
from datetime import datetime, timezone, timedelta

SURGE_LANDING = LAKEHOUSE / "marketplace" / "surge_inference_landing"
SURGE_LANDING.mkdir(parents=True, exist_ok=True)

random.seed(42)
cities = ["LON", "MAN", "BIR", "LDS", "BRS", "EDI", "GLA"]
n = 200

surge_df = pl.DataFrame(
    {
        "inference_id": [str(uuid.uuid4()) for _ in range(n)],
        "model_version": [random.choice(["v2.1.0", "v2.2.0", "v2.3.0-beta"]) for _ in range(n)],
        "city_code": [random.choice(cities) for _ in range(n)],
        "demand_index": [round(random.uniform(0.5, 8.5), 2) for _ in range(n)],
        "surge_multiplier": [round(random.uniform(1.0, 4.2), 2) for _ in range(n)],
        "timestamp": [datetime.now(timezone.utc) - timedelta(minutes=random.randint(0, 1440)) for _ in range(n)],
    }
)

surge_df.write_parquet(SURGE_LANDING / "inferences.parquet")
print(f"✅ Generated {n} surge inference records → {SURGE_LANDING}")
surge_df.head(3)

✅ Generated 200 surge inference records → lakehouse\marketplace\surge_inference_landing


inference_id,model_version,city_code,demand_index,surge_multiplier,timestamp
str,str,str,f64,f64,"datetime[μs, UTC]"
"""907921d4-ba58-40ef-8f48-677566e12f29""","""v2.3.0-beta""","""LON""",5.86,2.42,2026-04-30 02:38:40.129519 UTC
"""42c655c9-5a77-4673-85cd-9c50f1e033ca""","""v2.1.0""","""MAN""",3.01,3.61,2026-04-30 15:53:40.129519 UTC
"""e8cd6736-db7e-49a1-badb-dfb1904d8734""","""v2.1.0""","""BRS""",2.62,3.85,2026-04-30 18:58:40.129519 UTC


In [14]:
print("Starting Gold Processing via LakeLogic Pipeline Driver...\n")

summary = runner.run(
    target_layers="gold",
    reload_layers=reload_layers,
    dry_run=False,
    environment=ENV,
    retry_attempts=retry_attempts,
    retry_base_wait_seconds=retry_base_wait_seconds,
    parallel=parallel,
)
print(summary)

2026-04-30 20:23:40.153 | INFO     | lakelogic.pipeline.runner:run:1505 - Pipeline storage mode: direct
2026-04-30 20:23:40.157 | INFO     | lakelogic.pipeline.runner:run:1606 - ── Processing Layer: GOLD (4 contracts) ──
2026-04-30 20:23:40.158 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2069 -   ─────────────────────────────────────────────────────────
2026-04-30 20:23:40.158 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2070 -   📄 [gold] gold_rideflow_trip_kpis | Contract: Gold — Trip KPIs v1.0.0
2026-04-30 20:23:40.174 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\silver\silver_rideflow_trips via duckdb
2026-04-30 20:23:40.233 | INFO     | lakelogic.core.processor:run_source:1815 - Incremental load: first run or target empty — loading all rows.


Starting Gold Processing via LakeLogic Pipeline Driver...



2026-04-30 20:23:40.411 | INFO     | lakelogic.engines.duckdb:_register_links:172 - Registered link 'silver_trips_full' from lakehouse\marketplace\silver\silver_rideflow_trips (type=delta)
2026-04-30 20:23:40.826 | INFO     | lakelogic.core.processor:run:818 - Run complete [domain=marketplace, system=rideflow, layer=gold] | Source: 1718 | Total: 19 | Good: 19 | Quarantine: 0 | Aggregated: 1699 | Ratio: 0.00%
2026-04-30 20:23:40.848 | INFO     | lakelogic.core.materialization:materialize_dataframe:3686 - 
Casting 6 column(s) to match contract types before Delta write (new table):
Column Name                         | From Type       | To Type        
-----------------------------------------------------------------------
kpi_date                            | timestamp[us]   | date32[day]    
total_trips                         | int64           | int32          
total_revenue                       | double          | float          
total_distance_km                   | double          

 PIPELINE RUN SUMMARY
  Pipeline run    : c6bcb99d-a869-41c8-a3b2-69098a1abaf0
  Environment     : local
  Dry run         : False

  Table Name                             Layer    Status     Rows     Good/Qrtn   
  ------------------------------------------------------------------------------
  gold_rideflow_trip_kpis                gold     success    19       19/0        
  gold_rideflow_driver_scorecard         gold     success    822      822/0       
  gold_rideflow_rider_daily_metrics      gold     success    1135     1135/0      
  gold_rideflow_surge_pricing_model      gold     success    200      200/0       


---
## Step 7 · Quick Data Preview

Inspect the Silver trips table — the main fact table driving the dashboard.

In [15]:
trips_path = LAKEHOUSE / registry.domain / registry.silver_layer / "silver_rideflow_trips"

if trips_path.exists():
    try:
        trips_df = pl.read_delta(str(trips_path))
    except Exception:
        parquets = list(trips_path.rglob("*.parquet"))
        trips_df = pl.read_parquet(parquets[0]) if parquets else pl.DataFrame()

    print(f"Silver trips: {len(trips_df):,} rows, {len(trips_df.columns)} columns")
    print(f"\nColumns: {trips_df.columns}")
    print("\nSample (5 rows):")
    display(trips_df.head(5))

    # Quick stats
    if "fare_amount" in trips_df.columns:
        fare_col = trips_df["fare_amount"].cast(pl.Float64, strict=False)
        print(f"\n📊 Fare stats:  min=£{fare_col.min():.2f}  avg=£{fare_col.mean():.2f}  max=£{fare_col.max():.2f}")
    if "city_code" in trips_df.columns:
        print(f"🏙️ Cities: {trips_df['city_code'].unique().sort().to_list()}")
else:
    print("⚠️ Silver trips table not found — check the pipeline cells above.")

Silver trips: 1,718 rows, 26 columns

Columns: ['trip_id', 'rider_id', 'driver_id', 'trip_type', 'pickup_lat', 'pickup_lng', 'dropoff_lat', 'dropoff_lng', 'city_code', 'requested_at', 'pickup_at', 'dropoff_at', 'distance_km', 'duration_minutes', 'fare_amount', 'surge_multiplier', 'tip_amount', 'payment_method', 'rider_rating', 'driver_rating', 'notes', '_lakelogic_source', '_lakelogic_processed_at', '_lakelogic_run_id', '_lakelogic_created_at', '_lakelogic_created_by']

Sample (5 rows):


trip_id,rider_id,driver_id,trip_type,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,city_code,requested_at,pickup_at,dropoff_at,distance_km,duration_minutes,fare_amount,surge_multiplier,tip_amount,payment_method,rider_rating,driver_rating,notes,_lakelogic_source,_lakelogic_processed_at,_lakelogic_run_id,_lakelogic_created_at,_lakelogic_created_by
str,str,str,str,str,str,str,str,str,datetime[μs],datetime[μs],datetime[μs],f32,i32,f32,f32,f32,str,f32,f32,str,str,str,str,str,str
"""18149762-ccea-48cf-8f76-da6cb2f364e6""","""R-8d318cae""","""D-1cae6186""","""eats_delivery""","""51.5525""","""0.016757""","""51.433948""","""-0.169965""","""XYZ""",2026-04-15 20:58:00,2026-04-15 21:05:00,2026-04-15 21:15:00,9.4,19,11.91,3.47,1.12,"""cash""",4.0,4.0,"""""","""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_trip_completed""","""2026-04-30T19:23:39+00:00""","""19b39783-a4f6-43d1-af4c-628f9d0c0ee4""","""2026-04-30T19:23:39+00:00""","""colli"""
"""d8195fff-e07a-4490-8406-3564eccc0b72""","""R-9075b324""","""D-019c3935""","""eats_delivery""","""51.432549""","""-0.035967""","""51.550409""","""-0.068297""","""XYZ""",2026-04-15 18:06:00,2026-04-15 18:09:00,2026-04-15 18:43:00,15.8,42,48.200001,2.09,3.99,"""apple_pay""",3.0,3.0,"""""","""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_trip_completed""","""2026-04-30T19:23:39+00:00""","""19b39783-a4f6-43d1-af4c-628f9d0c0ee4""","""2026-04-30T19:23:39+00:00""","""colli"""
"""75403d43-1540-4781-951e-6aba320840d0""","""R-a31b6328""","""D-7f57c6d9""","""eats_delivery""","""51.534781""","""-0.063942""","""51.464087""","""-0.146983""","""LON""",2026-04-15 05:00:00,2026-04-15 05:10:00,2026-04-15 05:52:00,18.299999,22,75.860001,5.8,2.97,"""google_pay""",1.0,5.0,"""""","""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_trip_completed""","""2026-04-30T19:23:39+00:00""","""19b39783-a4f6-43d1-af4c-628f9d0c0ee4""","""2026-04-30T19:23:39+00:00""","""colli"""
"""8ca1feeb-8d31-4e73-897b-09661d608ac8""","""R-7d9d7564""","""D-8fb3a9a3""","""eats_delivery""","""51.456935""","""0.022878""","""51.561762""","""-0.076129""","""LON""",2026-04-15 12:15:00,2026-04-15 12:00:00,2026-04-15 11:00:00,21.5,16,15.25,1.28,4.55,"""cash""",3.0,5.0,"""""","""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_trip_completed""","""2026-04-30T19:23:39+00:00""","""19b39783-a4f6-43d1-af4c-628f9d0c0ee4""","""2026-04-30T19:23:39+00:00""","""colli"""
"""b198ce6e-e63c-4411-b1f9-6c432a421f7f""","""R-d79f5920""","""D-557f520f""","""ride""","""51.437188""","""-0.169318""","""51.453839""","""0.014559""","""PAR""",2026-04-15 07:01:00,2026-04-15 12:00:00,2026-04-15 11:00:00,18.5,19,11.76,1.04,2.17,"""cash""",3.0,4.0,"""""","""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\bronze\bronze_rideflow_trip_completed""","""2026-04-30T19:23:39+00:00""","""19b39783-a4f6-43d1-af4c-628f9d0c0ee4""","""2026-04-30T19:23:39+00:00""","""colli"""



📊 Fare stats:  min=£1.87  avg=£39.17  max=£142.58
🏙️ Cities: ['BER', 'LON', 'NYC', 'PAR', 'SYD', 'TYO', 'XYZ']


---
## Step 8 · Pipeline Audit Trail

Every pipeline run is logged with row counts, durations, quarantine ratios, and estimated costs.

In [16]:
log_path = LAKEHOUSE / registry.domain / "_logs"

column_list = [
    "run_id",
    "contract",
    "dataset",
    "status",
    "timestamp",
    "run_duration_seconds",
    "engine",
    "counts_source",
    "counts_total",
    "counts_good",
    "counts_quarantined",
    "quarantine_ratio",
    "estimated_cost",
    "cost_currency",
]
if log_path.exists():
    try:
        from deltalake import DeltaTable

        logs_df = pl.from_arrow(DeltaTable(str(log_path)).to_pyarrow_table())
        print(f"Run logs: {len(logs_df)} entries")
        display(logs_df.select("*").head(2))
    except ImportError:
        logs_df = pl.read_delta(str(log_path))
        print(f"Run logs: {len(logs_df)} entries")
        display(logs_df.select(column_list).head(10))
    except Exception as e:
        print(f"Run log read error: {e}")
else:
    print("No run log yet — this is expected for the first pipeline run.")

Run logs: 15 entries


pipeline_run_id,run_id,timestamp,start_time,end_time,run_duration_seconds,engine,contract,stage,dataset,domain,system,environment,data_layer,status,error_message,source_path,counts_source,counts_total,counts_good,counts_quarantined,quarantine_ratio,estimated_cost,cost_currency,cost_confidence,max_source_mtime,max_watermark_value,dlt_state_json,slo_json,report_json
str,str,str,str,str,f64,str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,f64,f64,str,str,f64,str,str,str,str
"""c6bcb99d-a869-41c8-a3b2-69098a1abaf0""","""27c25e97-ecab-46f4-8aaf-f96f820b0280""","""2026-04-30T19:23:42+00:00""","""2026-04-30T19:23:42.866593+00:00""","""2026-04-30T19:23:42.929234+00:00""",0.062066,"""duckdb""","""Gold — Surge Pricing ML Model Log""","""default""","""gold_rideflow_surge_pricing_model""","""marketplace""","""rideflow""","""local""","""gold""","""succeeded""",null,"""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\surge_inference_landing""",200,200,200,0,0.0,0.0,"""USD""","""none""",1.7776e9,null,null,"""{""freshness"": {""seconds"": null, ""pass"": null, ""threshold_seconds"": null, ""source_seconds"": null, ""source_pass"": null}, ""availability"": {""ratio"": null, ""pass"": null, ""threshold"": null}, ""row_count"": {""…","""{""run_id"": ""27c25e97-ecab-46f4-8aaf-f96f820b0280"", ""pipeline_run_id"": ""c6bcb99d-a869-41c8-a3b2-69098a1abaf0"", ""engine"": ""duckdb"", ""contract"": ""Gold \u2014 Surge Pricing ML Model Log"", ""contract_file_n…"
"""c6bcb99d-a869-41c8-a3b2-69098a1abaf0""","""412003cb-38db-4504-a83c-c00ac7846fa2""","""2026-04-30T19:23:42+00:00""","""2026-04-30T19:23:42.076573+00:00""","""2026-04-30T19:23:42.635183+00:00""",0.55838,"""duckdb""","""Gold — Daily Rider Metrics""","""default""","""gold_rideflow_rider_daily_metrics""","""marketplace""","""rideflow""","""local""","""gold""","""succeeded""",null,"""C:\_Personal\_SaaS\lakelogic\examples\colab\lakehouse\marketplace\silver\silver_rideflow_trips""",1718,1135,1135,0,0.0,0.0,"""USD""","""none""",1.7776e9,null,null,"""{""freshness"": {""seconds"": null, ""pass"": null, ""threshold_seconds"": null, ""source_seconds"": null, ""source_pass"": null}, ""availability"": {""ratio"": null, ""pass"": null, ""threshold"": null}, ""row_count"": {""…","""{""run_id"": ""412003cb-38db-4504-a83c-c00ac7846fa2"", ""pipeline_run_id"": ""c6bcb99d-a869-41c8-a3b2-69098a1abaf0"", ""engine"": ""duckdb"", ""contract"": ""Gold \u2014 Daily Rider Metrics"", ""contract_file_name"": n…"


---
## Step 8b · Quarantine View

In [17]:
table_name = "gold_rideflow_rider_lifetime_value"
quarantine_path = LAKEHOUSE / registry.domain / "_quarantine" / table_name

if quarantine_path.exists():
    try:
        from deltalake import DeltaTable

        quarantines_df = pl.from_arrow(DeltaTable(str(quarantine_path)).to_pyarrow_table())
        print(f"quarantine quarantines: {len(quarantines_df)} entries")
        display(quarantines_df.select("*").head(2))
    except ImportError:
        quarantines_df = pl.read_delta(str(quarantine_path))
        print(f"Run quarantines: {len(quarantines_df)} entries")
        display(quarantines_df.select(column_list).head(10))
    except Exception as e:
        print(f"Run quarantine read error: {e}")
else:
    print("No run quarantine yet — this is expected for the first pipeline run.")

No run quarantine yet — this is expected for the first pipeline run.


---
## Step 9 · 🚗 Launch Live Dashboard

The payoff! A real-time Panel dashboard that auto-refreshes from the Silver Delta tables every 3 seconds.

**5 Analytical Views:**
- 📈 Trip Volume Timeline (with Hour / Day / Week / Month aggregation)
- ⭐ Top 10 Drivers by Revenue (cross-table join with driver profiles)
- 🏙️ Revenue by City
- 👍 Driver Quality Distribution
- ⚡ Surge Multiplier Profile

> **Tip:** Use `dashboard.show()` to pop the dashboard out into a full browser tab for the best experience.

In [18]:
import importlib
import assets.dashboards.shared.streaming_dashboard

importlib.reload(assets.dashboards.shared.streaming_dashboard)

<module 'assets.dashboards.shared.streaming_dashboard' from 'c:\\_Personal\\_SaaS\\lakelogic\\examples\\colab\\assets\\dashboards\\shared\\streaming_dashboard.py'>

In [19]:
sys.path.insert(0, ".")
from assets.dashboards.shared.streaming_dashboard import create_streaming_dashboard_inline

dashboard = create_streaming_dashboard_inline(
    lakehouse_root=str(LAKEHOUSE),
    domain=registry.domain,
    system=registry.system,
    refresh_ms=3000,
)
dashboard

BokehModel(combine_events=True, render_bundle={'docs_json': {'899202a1-9ebe-464e-954e-0a5358d018b7': {'version…

---
## Step 10 · 🦆 Build Local DuckDB Catalog

Create a persistent `lakehouse_catalog.duckdb` file that maps all your Delta tables as SQL views.
Open this file in **DBeaver** (DuckDB driver) to browse and query your entire lakehouse with standard SQL.

```sql
-- Once opened in DBeaver, just run:
SELECT * FROM silver_trips WHERE city_code = 'LON' LIMIT 100;
```

In [20]:
import duckdb

catalog_path = str(LAKEHOUSE / f"lakehouse_catalog_{registry.domain}.duckdb")
con = duckdb.connect(catalog_path)

con.execute("INSTALL delta; LOAD delta;")

# Build views for every Delta table in the lakehouse
domain_root = LAKEHOUSE / registry.domain
for layer in ["bronze", "silver", "gold", "quarantine", "_logs"]:
    layer_path = domain_root / layer
    if not layer_path.exists():
        continue
    for table_dir in sorted(layer_path.iterdir()):
        if table_dir.is_dir() and not table_dir.name.startswith("_"):
            view_name = table_dir.name
            try:
                con.execute(f"CREATE OR REPLACE VIEW {view_name} AS SELECT * FROM delta_scan('{table_dir.as_posix()}')")
                row_count = con.execute(f"SELECT COUNT(*) FROM {view_name}").fetchone()[0]
                print(f"  ✅ {view_name}: {row_count:,} rows")
            except Exception as e:
                print(f"  ⚠️ {view_name}: {e}")

con.close()
print(f"\n🦆 Catalog saved → {catalog_path}")
print("   Open this file in DBeaver (DuckDB driver) to browse your lakehouse!")

  ✅ bronze_rideflow_driver_profiles: 148 rows
  ✅ bronze_rideflow_driver_telemetry: 7,916 rows
  ✅ bronze_rideflow_rider_app_events: 2,869 rows
  ✅ bronze_rideflow_rider_profiles: 391 rows
  ✅ bronze_rideflow_trip_cancellations: 179 rows
  ✅ bronze_rideflow_trip_completed: 1,838 rows
  ✅ bronze_rideflow_trip_requests: 1,453 rows
  ✅ silver_rideflow_driver_profiles: 148 rows
  ✅ silver_rideflow_driver_telemetry: 7,916 rows
  ✅ silver_rideflow_rider_profiles: 391 rows
  ✅ silver_rideflow_trips: 1,718 rows
  ✅ gold_rideflow_driver_scorecard: 822 rows
  ✅ gold_rideflow_rider_daily_metrics: 1,135 rows
  ✅ gold_rideflow_surge_pricing_model: 200 rows
  ✅ gold_rideflow_trip_kpis: 19 rows

🦆 Catalog saved → lakehouse\lakehouse_catalog_marketplace.duckdb
   Open this file in DBeaver (DuckDB driver) to browse your lakehouse!


---
## 🔄 Bonus: Incremental Streaming Mode

Re-run the simulator for the **next 7 hours** of synthetic data and trigger a full pipeline pass after each window.
This simulates what would happen in production with **Databricks Structured Streaming** — new micro-batches
land every hour, and the pipeline incrementally ingests them through Bronze → Silver → Gold.

| Parameter | Value |
| :-- | :-- |
| Simulation horizon | 7 hours from now |
| Window cadence | 60 min (1 window = 1 simulated hour) |
| Inter-window pause | 10 s (adjustable — set higher to watch dashboard animate) |
| Pipeline layers | Bronze → Silver → Gold (full pass each window) |

In [21]:
import time
from datetime import datetime, timedelta, timezone

if enable_incremental_streaming:
    # ── Configuration ─────────────────────────────────────────────────
    STREAM_HOURS = 7  # Simulate 7 hours into the future
    PAUSE_SECONDS = 10  # Pause between windows (increase to watch dashboard animate)
    MICRO_BATCHES = 10  # Files per entity per window

    stream_until = datetime.now(timezone.utc) + timedelta(hours=STREAM_HOURS)

    print("🔄 Incremental Streaming Mode")
    print(f"   Generating {STREAM_HOURS} hours of future data → up to {stream_until.strftime('%Y-%m-%d %H:%M UTC')}")
    print(f"   Pause between windows: {PAUSE_SECONDS}s")
    print(f"   Pipeline engine: {engine}")
    print(f"{'=' * 70}")

    cumulative_rows = 0
    window_count = 0
    t_start = time.time()

    for window in sim.run(
        num_windows=STREAM_HOURS,
        include_seed=False,
        micro_batches=MICRO_BATCHES,
        up_to=stream_until,
        resume=True,
    ):
        window_count += 1
        cumulative_rows += window.total_rows

        print(
            f"\n⏱  Window {window_count}/{STREAM_HOURS} | "
            f"{window.timestamp.strftime('%Y-%m-%d %H:%M')} | "
            f"+{window.total_rows:,} rows  (cumulative: {cumulative_rows:,})"
        )

        # ── Re-ingest through the full pipeline ───────────────────────
        print("   🏗  Running pipeline (Bronze → Silver → Gold)...")
        runner = LakehousePipeline(registry, engine=engine)
        summary = runner.run(
            target_layers="bronze,silver,gold",
            retry_attempts=retry_attempts,
            retry_base_wait_seconds=retry_base_wait_seconds,
            parallel=parallel,
        )

        # Quick summary of what changed
        ingested = [r for r in summary.results if r.get("status") == "success"]
        skipped = [r for r in summary.results if r.get("status") in ("no_new_data", "no_new_rows")]
        print(f"   ✅ Pipeline pass complete: {len(ingested)} ingested, {len(skipped)} skipped")

        # ── Pause before next window ──────────────────────────────────
        if window_count < STREAM_HOURS:
            print(f"   ⏸  Pausing {PAUSE_SECONDS}s before next window...")
            time.sleep(PAUSE_SECONDS)

    elapsed = time.time() - t_start
    print(f"\n{'=' * 70}")
    print("✅ Incremental streaming complete!")
    print(f"   Windows processed: {window_count}")
    print(f"   Total rows generated: {cumulative_rows:,}")
    print(f"   Elapsed wall time: {elapsed / 60:.1f} min")
    print(f"   Dashboard should now reflect {STREAM_HOURS} additional hours of data.")
else:
    print("⏭  Incremental streaming disabled (set enable_incremental_streaming = True to activate)")

⏭  Incremental streaming disabled (set enable_incremental_streaming = True to activate)


---
## ✅ Data Products Published

This domain pipeline has successfully generated and published the following data products to the shared lakehouse:

| Data Product | Consumers |
| :-- | :-- |
| `silver_rideflow_trips` | Payments (reconciliation), Operations (support enrichment) |
| `silver_rideflow_driver_profiles` | Operations (background checks), Marketing (attribution) |
| `gold_rideflow_trip_kpis` | Executive Dashboard, Mesh Products |
| `gold_rideflow_rider_lifetime_value` | Marketing (LTV attribution), Finance (forecasting) |

Other domain pipelines (`08_rideflow_payments`, `09_rideflow_operations`, `10_rideflow_marketing`) read from these outputs — never from our internal Bronze tables. This enforces clean Data Mesh boundaries.

### Next Notebooks

- **`08_rideflow_payments.ipynb`** — Payments domain pipeline (Stripe charges, payouts, reconciliation)
- **`12_compliance_gdpr_rtbf.ipynb`** — Execute a Right-to-Be-Forgotten request across all domains
- **`14_data_mesh_dashboards.ipynb`** — Live observability across all domain pipelines